# 00 TT-Conv：表現とdense kernel再構成

小さい`Conv2d`の重みを1つのTT-matrix表現へ写し、TT coresからdense kernelを再構成します。

このNotebookの中心は、**Conv kernelの要素を壊さず行列化・tensorizeし、exact TTと低rank TTを区別して検証すること**です。学習済みCNN、rank sweep、tensorization比較、fine-tuning、直接TT contractionはまだ扱いません。


## 1. このNotebookで確認すること

1. Conv weight $W_{o,c,u,v}$を行列$W_{\mathrm{mat}}$へ写す。
2. 1つのtensorizationを固定し、TT core shapeとactual rankを確認する。
3. exact TTとrankを制限したTTからdense kernelを再構成する。
4. 元kernelとの近似誤差と、Conv出力の誤差を分けて測る。
5. 再構成経路を通して全TT coreへgradientが流れることを確認する。
6. exact reconstructionとparameter圧縮が別の性質であることを数値で確認する。

### このNotebookでは行わないこと

- このtensorizationが最適だとは主張しない。
- dense kernelを作らないTT-Conv forwardは実装しない。
- Fashion-MNISTのdatasetや学習済みCNNは読み込まない。
- MACs、latency、accuracy、fine-tuningは評価しない。


## 2. importと再現性

TT分解、dense再構成、rank、parameter数には既存APIを使用します。Notebook内で同じ関数を再実装しません。


In [40]:
from __future__ import annotations
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import math
import sys
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from torch import nn

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / "src"
    if (src_dir / "nn_compression").is_dir():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        project_root = candidate
        break
else:
    raise RuntimeError("src/nn_compressionが見つかりません。")

from nn_compression.compression import (
    dense_to_tt_matrix_cores,
    dense_to_tt_matrix_tensor,
    tt_matrix_num_parameters,
    tt_matrix_ranks,
    tt_matrix_to_dense,
)
from nn_compression.metrics import relative_frobenius_error
from nn_compression.utils import set_seed

SEED = 42
DTYPE = torch.float64
DEVICE = torch.device("cpu")
RTOL = 1e-9
ATOL = 1e-10
set_seed(SEED)

print("project root:", project_root)
print("torch:", torch.__version__)
print("device / dtype:", DEVICE, DTYPE)


project root: D:\dev\nn-compression-svd-dmrg
torch: 2.11.0+cu128
device / dtype: cpu torch.float64


## 3. 小さいConv2dを固定する

このNotebookでは次の条件だけを扱います。

- $C_{\mathrm{in}}=4$
- $C_{\mathrm{out}}=8$
- kernel size $=3\times3$
- stride $=1$
- padding $=1$
- dilation $=1$
- groups $=1$
- biasあり

weight shapeは`(8, 4, 3, 3)`、1つの出力channelに対する受容野の要素数は$4\times3\times3=36$です。


In [41]:
BATCH_SIZE = 2
IN_CHANNELS = 4
OUT_CHANNELS = 8
KERNEL_SIZE = (3, 3)
STRIDE = (1, 1)
PADDING = (1, 1)
DILATION = (1, 1)
GROUPS = 1
INPUT_HW = (7, 7)

dense_conv = nn.Conv2d(
    in_channels=IN_CHANNELS,
    out_channels=OUT_CHANNELS,
    kernel_size=KERNEL_SIZE,
    stride=STRIDE,
    padding=PADDING,
    dilation=DILATION,
    groups=GROUPS,
    bias=True,
    dtype=DTYPE,
    device=DEVICE,
)
input_tensor = torch.randn(
    BATCH_SIZE, IN_CHANNELS, *INPUT_HW, dtype=DTYPE, device=DEVICE
)
reference_output = dense_conv(input_tensor)

assert tuple(dense_conv.weight.shape) == (8, 4, 3, 3)
assert tuple(dense_conv.bias.shape) == (8,)
assert tuple(reference_output.shape) == (2, 8, 7, 7)
print("weight shape:", tuple(dense_conv.weight.shape))
print("input shape :", tuple(input_tensor.shape))
print("output shape:", tuple(reference_output.shape))


weight shape: (8, 4, 3, 3)
input shape : (2, 4, 7, 7)
output shape: (2, 8, 7, 7)


## 4. 今回採用するTT表現

Conv kernelを、出力channelと1受容野の入力を結ぶ行列として扱います。PyTorchのweight配置を保ったまま、

$$
W\in\mathbb{R}^{C_{\mathrm{out}}\times C_{\mathrm{in}}\times K_H\times K_W}
\longrightarrow
W_{\mathrm{mat}}\in\mathbb{R}^{C_{\mathrm{out}}\times(C_{\mathrm{in}}K_HK_W)}
$$

と`reshape`します。今回のshapeは$8\times36$です。

3-core TT-matrixとして、

$$
(m_1,m_2,m_3)=(2,2,2),\qquad (n_1,n_2,n_3)=(3,3,4)
$$

を固定します。したがって各coreは、

$$
G^{(k)}\in\mathbb{R}^{r_{k-1}\times m_k\times n_k\times r_k}
$$

というshapeを持ちます。これは空間軸を独立coreにする方式ではなく、$C_{\mathrm{in}}K_HK_W$をまとめてfactorizeする1つの選択です。最適性は主張しません。


In [42]:
OUT_MODES = (2, 2, 2)
IN_MODES = (3, 3, 4)
TRUNCATED_MAX_RANK = 2

assert math.prod(OUT_MODES) == OUT_CHANNELS
assert math.prod(IN_MODES) == IN_CHANNELS * math.prod(KERNEL_SIZE)
assert dense_conv.weight.numel() == math.prod(OUT_MODES) * math.prod(IN_MODES)
print("out modes:", OUT_MODES)
print("in modes :", IN_MODES)
print("combined physical modes:", tuple(m * n for m, n in zip(OUT_MODES, IN_MODES)))


out modes: (2, 2, 2)
in modes : (3, 3, 4)
combined physical modes: (6, 6, 8)


## 5. TODO 1：Conv kernelを行列化し、TT-SVD入力を確認する

### このセルで確認すること

PyTorchのweight要素順を変えず、`(8, 4, 3, 3)`から`(8, 36)`へreshapeします。その後、既存API`dense_to_tt_matrix_tensor`でTT-SVD入力を作ります。

### 入力

- `dense_conv.weight`
- `OUT_MODES`
- `IN_MODES`

### 出力

- `weight_matrix: torch.Tensor`、shape `(8, 36)`
- `combined_tensor: torch.Tensor`、shape `(6, 6, 8)`

### このセルでは行わないこと

TT-SVD、kernel再構成、Conv forwardはまだ行いません。


In [43]:
# TODO 1-1: dense_conv.weightを(C_out, C_in * K_H * K_W)へreshapeする。
weight_matrix = dense_conv.weight.reshape(OUT_CHANNELS , IN_CHANNELS *KERNEL_SIZE[0]*KERNEL_SIZE[1])

# TODO 1-2: 既存APIでTT-SVD入力用combined tensorを作る。
combined_tensor =dense_to_tt_matrix_tensor(weight_matrix,OUT_MODES,IN_MODES)


In [44]:
todo1_names = ("weight_matrix", "combined_tensor")
todo1_missing = [name for name in todo1_names if name not in globals()]
if todo1_missing:
    print("TODO 1未実装:", todo1_missing)
else:
    assert isinstance(weight_matrix, torch.Tensor)
    assert isinstance(combined_tensor, torch.Tensor)
    assert tuple(weight_matrix.shape) == (8, 36)
    assert tuple(combined_tensor.shape) == (6, 6, 8)
    assert weight_matrix.numel() == dense_conv.weight.numel()
    assert weight_matrix.data_ptr() == dense_conv.weight.data_ptr()
    print("TODO 1 checks: OK")


TODO 1 checks: OK


## 6. TODO 2：exact TTとrank制限TTを作る

### 数式上の違い

exact TTでは各unfoldingの非零特異値を保持します。rank制限TTでは、各内部bondについて、

$$
r_k\leq r_{\max}=2
$$

とします。

### 使用する既存関数

`dense_to_tt_matrix_cores(weight_matrix, OUT_MODES, IN_MODES, max_rank=...)`を使います。TT-SVDをNotebook内で再実装しません。

### 出力

- `exact_cores: list[torch.Tensor]`
- `truncated_cores: list[torch.Tensor]`


In [45]:
# TODO 2-1: 打ち切りなしのexact TT-matrix coresを作る。
# dense_to_tt_matrix_cores の第1引数は dense 行列 (weight_matrix)。
# combined_tensor は渡さない（内部で dense_to_tt_matrix_tensor が走る）。
exact_cores = dense_to_tt_matrix_cores(
    weight_matrix, OUT_MODES, IN_MODES, max_rank=None
)

# TODO 2-2: max_rank=TRUNCATED_MAX_RANKのTT-matrix coresを作る。
truncated_cores = dense_to_tt_matrix_cores(
    weight_matrix, OUT_MODES, IN_MODES, max_rank=TRUNCATED_MAX_RANK
)


In [46]:
todo2_names = ("exact_cores", "truncated_cores")
todo2_missing = [name for name in todo2_names if name not in globals()]
if todo2_missing:
    print("TODO 2未実装:", todo2_missing)
else:
    assert len(exact_cores) == len(OUT_MODES) == 3
    assert len(truncated_cores) == len(OUT_MODES) == 3
    exact_ranks = tt_matrix_ranks(exact_cores)
    truncated_ranks = tt_matrix_ranks(truncated_cores)
    assert exact_ranks[0] == exact_ranks[-1] == 1
    assert truncated_ranks[0] == truncated_ranks[-1] == 1
    assert max(truncated_ranks) <= TRUNCATED_MAX_RANK
    print("exact ranks   :", exact_ranks)
    print("truncated ranks:", truncated_ranks)
    print("exact core shapes   :", [tuple(core.shape) for core in exact_cores])
    print("truncated core shapes:", [tuple(core.shape) for core in truncated_cores])


exact ranks   : (1, 6, 8, 1)
truncated ranks: (1, 2, 2, 1)
exact core shapes   : [(1, 2, 3, 6), (6, 2, 3, 8), (8, 2, 4, 1)]
truncated core shapes: [(1, 2, 3, 2), (2, 2, 3, 2), (2, 2, 4, 1)]


## 7. TODO 3：dense kernelを再構成する

既存API`tt_matrix_to_dense`はshape `(8, 36)`の行列を返します。これを元のConv weight shape `(8, 4, 3, 3)`へreshapeします。

### 出力

- `exact_weight_matrix`と`exact_kernel`
- `truncated_weight_matrix`と`truncated_kernel`

exact kernelは元kernelと数値誤差内で一致するはずです。truncated kernelは近似なので、完全一致を要求しません。


In [47]:
# TODO 3-1: exact coresからweight matrixを再構成し、Conv kernel shapeへ戻す。
exact_weight_matrix = tt_matrix_to_dense(exact_cores,OUT_MODES,IN_MODES)
exact_kernel = exact_weight_matrix.reshape(OUT_CHANNELS , IN_CHANNELS ,KERNEL_SIZE[0],KERNEL_SIZE[1])

# TODO 3-2: truncated coresについても同じ処理を行う。
truncated_weight_matrix = tt_matrix_to_dense(truncated_cores,OUT_MODES,IN_MODES)
truncated_kernel = truncated_weight_matrix.reshape(OUT_CHANNELS , IN_CHANNELS ,KERNEL_SIZE[0],KERNEL_SIZE[1])


In [48]:
todo3_names = (
    "exact_weight_matrix", "exact_kernel",
    "truncated_weight_matrix", "truncated_kernel",
)
todo3_missing = [name for name in todo3_names if name not in globals()]
if todo3_missing:
    print("TODO 3未実装:", todo3_missing)
else:
    assert tuple(exact_weight_matrix.shape) == (8, 36)
    assert tuple(truncated_weight_matrix.shape) == (8, 36)
    assert tuple(exact_kernel.shape) == tuple(dense_conv.weight.shape)
    assert tuple(truncated_kernel.shape) == tuple(dense_conv.weight.shape)
    torch.testing.assert_close(
        exact_kernel, dense_conv.weight, rtol=RTOL, atol=ATOL
    )
    exact_relative_error = float(
        relative_frobenius_error(dense_conv.weight, exact_kernel).detach()
    )
    truncated_relative_error = float(
        relative_frobenius_error(dense_conv.weight, truncated_kernel).detach()
    )
    assert exact_relative_error <= 1e-8
    assert truncated_relative_error >= exact_relative_error
    print("exact relative error   :", exact_relative_error)
    print("truncated relative error:", truncated_relative_error)


exact relative error   : 2.4658159220156266e-15
truncated relative error: 0.8837186876462422


## 8. Conv forwardを比較する

`torch.nn.functional.conv2d`を使い、元kernel、exact再構成kernel、truncated再構成kernelで同じ入力を畳み込みます。stride、padding、dilation、groups、biasは元の`dense_conv`と揃えます。

ここで確認するのは次の2種類です。

- 元dense Convとexact再構成Convの一致：実装・要素対応の検証
- 元dense Convとtruncated再構成Convの差：圧縮近似誤差の観察

00ではまだdense kernelを明示的に作っています。直接TT contractionとの一致は01で扱います。ここはTT固有の新規実装ではないため、検証コードを完成済みにしています。


In [49]:
forward_dependencies = ("exact_kernel", "truncated_kernel")
forward_missing = [name for name in forward_dependencies if name not in globals()]
if forward_missing:
    print("TODO 3完了後に実行できます:", forward_missing)
else:
    exact_output = F.conv2d(
        input_tensor, exact_kernel, dense_conv.bias,
        stride=STRIDE, padding=PADDING, dilation=DILATION, groups=GROUPS,
    )
    truncated_output = F.conv2d(
        input_tensor, truncated_kernel, dense_conv.bias,
        stride=STRIDE, padding=PADDING, dilation=DILATION, groups=GROUPS,
    )
    exact_output_rmse = torch.sqrt(torch.mean((exact_output - reference_output) ** 2))
    truncated_output_rmse = torch.sqrt(
        torch.mean((truncated_output - reference_output) ** 2)
    )


In [50]:
todo4_names = (
    "exact_output", "truncated_output",
    "exact_output_rmse", "truncated_output_rmse",
)
todo4_missing = [name for name in todo4_names if name not in globals()]
if todo4_missing:
    print("forward比較はTODO 3完了後に実行できます:", todo4_missing)
else:
    assert tuple(exact_output.shape) == tuple(reference_output.shape)
    assert tuple(truncated_output.shape) == tuple(reference_output.shape)
    torch.testing.assert_close(
        exact_output, reference_output, rtol=RTOL, atol=ATOL
    )
    exact_output_rmse_value = float(exact_output_rmse.detach())
    truncated_output_rmse_value = float(truncated_output_rmse.detach())
    assert exact_output_rmse_value <= 1e-8
    assert truncated_output_rmse_value >= exact_output_rmse_value
    print("exact output RMSE   :", exact_output_rmse_value)
    print("truncated output RMSE:", truncated_output_rmse_value)


exact output RMSE   : 1.2349312793258565e-15
truncated output RMSE: 0.44025388243651964


## 9. TODO 4：再構成経路のgradientを確認する

truncated coresを`nn.ParameterList`として所有し直し、

$$
\{G^{(k)}\}\rightarrow W_{\mathrm{mat}}\rightarrow W_{\mathrm{conv}}\rightarrow y\rightarrow\mathcal{L}
$$

という計算グラフを作ります。lossは固定した乱数targetとの平均二乗誤差とします。

各coreについて、`grad is not None`、shape一致、finiteを必須条件にします。この固定例ではgradient normが正になることも確認しますが、任意の入力・lossに対する一般命題にはしません。

既存の`tt_matrix_to_dense`はcore列として`list`または`tuple`を受け取るため、所有は`nn.ParameterList`のまま、呼び出し時だけ`list(learnable_cores)`へ変換します。Tensor自体はコピーされず、autograd graphも維持されます。


In [51]:
set_seed(SEED + 1)
gradient_target = torch.randn_like(reference_output)


In [52]:
# TODO 4-1: truncated_coresをdetach().clone()し、nn.ParameterListへ入れる。
learnable_cores: nn.ParameterList = nn.ParameterList([
    nn.Parameter(core.detach().clone())
    for core in truncated_cores
])
# TODO 4-2: list(learnable_cores)を既存APIへ渡し、weight matrixとConv kernelを再構成する。
learnable_weight_matrix = tt_matrix_to_dense(
    list(learnable_cores), OUT_MODES, IN_MODES
)
learnable_kernel = learnable_weight_matrix.reshape(OUT_CHANNELS , IN_CHANNELS ,KERNEL_SIZE[0],KERNEL_SIZE[1])


In [53]:
todo4_names = (
    "learnable_cores", "learnable_weight_matrix", "learnable_kernel",
)
todo4_missing = [name for name in todo4_names if name not in globals()]
if todo4_missing:
    print("TODO 4未実装:", todo4_missing)
else:
    assert isinstance(learnable_cores, nn.ParameterList)
    assert tuple(learnable_kernel.shape) == tuple(dense_conv.weight.shape)
    learnable_output = F.conv2d(
        input_tensor, learnable_kernel, dense_conv.bias,
        stride=STRIDE, padding=PADDING, dilation=DILATION, groups=GROUPS,
    )
    reconstruction_loss = F.mse_loss(learnable_output, gradient_target)
    for core in learnable_cores:
        core.grad = None
    reconstruction_loss.backward()
    gradient_records = [
        {
            "core_index": core_index,
            "core_shape": tuple(core.shape),
            "gradient_norm": float(torch.linalg.vector_norm(core.grad)),
        }
        for core_index, core in enumerate(learnable_cores)
    ]
    df_core_gradients = pd.DataFrame(gradient_records)
    assert tuple(learnable_output.shape) == tuple(reference_output.shape)
    assert torch.isfinite(reconstruction_loss)
    assert len(df_core_gradients) == len(learnable_cores)
    for core in learnable_cores:
        assert core.grad is not None
        assert core.grad.shape == core.shape
        assert torch.isfinite(core.grad).all()
        assert float(torch.linalg.vector_norm(core.grad)) > 0.0
    display(df_core_gradients)
    print("gradient checks: OK")


,core_index,core_shape,gradient_norm
0,0,"(1, 2, 3, 2)",0.117466
1,1,"(2, 2, 3, 2)",0.118728
2,2,"(2, 2, 4, 1)",0.220505


gradient checks: OK


## 10. TODO 5：parameter数を比較する

dense Convはweightとbiasを保存します。TT表現はTT coresと同じbiasを保存するとします。

$$
P_{\mathrm{dense}}=C_{\mathrm{out}}C_{\mathrm{in}}K_HK_W+C_{\mathrm{out}}
$$

$$
P_{\mathrm{TT}}=\sum_{k=1}^{d}r_{k-1}m_kn_kr_k+C_{\mathrm{out}}
$$

を比較します。TT coreの要素数には既存API`tt_matrix_num_parameters`を使用します。

exact reconstructionできても、TT parameter数がdense以下になるとは限りません。


In [54]:
dense_parameters = dense_conv.weight.numel() + dense_conv.bias.numel()

# TODO 5-1: exact TT coresとbiasの総parameter数を計算する。
exact_tt_parameters = tt_matrix_num_parameters(exact_cores) + dense_conv.bias.numel()

# TODO 5-2: truncated TT coresとbiasの総parameter数を計算する。
truncated_tt_parameters = tt_matrix_num_parameters(truncated_cores) + dense_conv.bias.numel()


In [55]:
todo5_names = ("exact_tt_parameters", "truncated_tt_parameters")
todo5_missing = [name for name in todo5_names if name not in globals()]
if todo5_missing:
    print("TODO 5未実装:", todo5_missing)
else:
    expected_dense_parameters = dense_conv.weight.numel() + dense_conv.bias.numel()
    assert dense_parameters == expected_dense_parameters
    assert exact_tt_parameters == (
        tt_matrix_num_parameters(exact_cores) + dense_conv.bias.numel()
    )
    assert truncated_tt_parameters == (
        tt_matrix_num_parameters(truncated_cores) + dense_conv.bias.numel()
    )
    assert exact_tt_parameters > dense_parameters
    assert truncated_tt_parameters < dense_parameters
    parameter_records = [
        {
            "representation": "dense",
            "actual_ranks": None,
            "parameters": dense_parameters,
            "compression_ratio": 1.0,
        },
        {
            "representation": "exact TT",
            "actual_ranks": tt_matrix_ranks(exact_cores),
            "parameters": exact_tt_parameters,
            "compression_ratio": dense_parameters / exact_tt_parameters,
        },
        {
            "representation": "rank-limited TT",
            "actual_ranks": tt_matrix_ranks(truncated_cores),
            "parameters": truncated_tt_parameters,
            "compression_ratio": dense_parameters / truncated_tt_parameters,
        },
    ]
    df_parameter_comparison = pd.DataFrame(parameter_records)
    assert len(df_parameter_comparison) == 3
    display(df_parameter_comparison)
    print("parameter checks: OK")


,representation,actual_ranks,parameters,compression_ratio
0,dense,None,296,1.000000
1,exact TT,"(1, 6, 8, 1)",396,0.747475
2,rank-limited TT,"(1, 2, 2, 1)",60,4.933333


parameter checks: OK


## 11. 実装後の確認項目

- [ ] `weight_matrix.shape == (8, 36)`
- [ ] `combined_tensor.shape == (6, 6, 8)`
- [ ] exact TTとtruncated TTのactual rankを別々に確認した
- [ ] exact kernelは元kernelと数値誤差内で一致した
- [ ] truncated kernelは近似であり、誤差を記録した
- [ ] exact再構成kernelのConv出力が元Conv出力と一致した
- [ ] 全TT coreのgradientが`None`ではなくfiniteだった
- [ ] exact reconstructionとparameter圧縮を別々に判断した

次のNotebookでは、dense kernelを明示的に再構成する経路を参照実装とし、TT coresから直接forwardする計算との一致を扱います。
